# HURDAT verification

In [3]:
"""
TC track and central-intensity verification for CorrDiff TC cases.

Pipeline
--------
1. Download + parse HURDAT2 best track (Atlantic).
2. For each cataloged storm, interpolate best-track position/intensity to the
   model valid hours.
3. Track every source field -- CONUS404 (WRF target), UNet baseline, and each
   CorrDiff member (ERA5- and GDAS-init) -- by locating the MSLP minimum within
   a search radius of the best-track center each hour.
4. Compute, per source/member/time:
     central intensity  = min MSLP in the core (+ max 10 m wind, secondary)
     track error        = great-circle distance (model center - reference center)
   against TWO references:
     HURDAT2   -> absolute track/intensity skill
     CONUS404  -> downscaling fidelity at fixed resolution
5. Aggregate per storm and pooled across all cases.

Method notes (read before trusting the numbers)
------------------------------------------------
* Central intensity is MIN MSLP (central pressure), which is resolution-tolerant
  and directly comparable to HURDAT2 Pmin. Max 10 m wind is reported too but is a
  weak Vmax analog: hourly MEAN wind != 1-min sustained, and 8 km under-resolves
  the peak -- so wind biases low, more so for intense storms. Treat it as
  secondary.
* The center is the MSLP minimum WITHIN R_km of the interpolated best-track
  position. This cannot detect a positional miss larger than R_km (returns the
  nearest local low instead). R_km is configurable.
* ENSEMBLE: every member is tracked separately and positions are aggregated
  AFTER. Never average the MSLP field and then find the min -- the average smears
  and shifts the low. Two distinct quantities are reported: the per-member track-
  error distribution, and the error of the ensemble-MEAN position (smaller when
  members straddle the truth).
* At 8 km both CONUS404 and CorrDiff run Pmin too high (weak) for intense storms;
  CONUS404-vs-HURDAT2 is the resolution "floor" -- compare CorrDiff to CONUS404
  to isolate downscaling skill from that floor.
"""
import os
import re
import warnings
import numpy as np
import xarray as xr


# Current Atlantic HURDAT2 (1851-2025, updated 2026-02-27; covers 2020-2025).
# The date suffix changes with each annual NHC revision -- see
# https://www.nhc.noaa.gov/data/hurdat/ for the latest.
HURDAT2_URL = "https://www.nhc.noaa.gov/data/hurdat/hurdat2-1851-2025-02272026.txt"

EARTH_R_KM = 6371.0


# ======================================================================
# HURDAT2 download + parse
# ======================================================================
def download_hurdat2(url=HURDAT2_URL, cache_path="hurdat2_atlantic.txt"):
    """Download HURDAT2 to a local cache (skip if present). Returns the text."""
    if os.path.exists(cache_path):
        with open(cache_path) as f:
            return f.read()
    import urllib.request
    print(f"downloading HURDAT2 -> {cache_path}")
    with urllib.request.urlopen(url, timeout=120) as r:
        text = r.read().decode("utf-8", errors="replace")
    with open(cache_path, "w") as f:
        f.write(text)
    return text


def parse_hurdat2(text):
    """
    Parse HURDAT2 text into {storm_id: dict(name, time, lat, lon, vmax, pmin)}.
    time is datetime64[m]; lat/lon in degrees (W,S negative); vmax kt; pmin mb.
    Missing intensities (-999 / -99) -> NaN.
    """
    storms = {}
    lines = [ln for ln in text.splitlines() if ln.strip()]
    i = 0
    hdr = re.compile(r"^[A-Z]{2}\d{6}$")
    while i < len(lines):
        f = [x.strip() for x in lines[i].split(",")]
        if hdr.match(f[0]):
            sid, name, n = f[0], f[1], int(f[2])
            t, lat, lon, vmax, pmin = [], [], [], [], []
            for r in lines[i + 1:i + 1 + n]:
                c = [x.strip() for x in r.split(",")]
                t.append(np.datetime64(
                    f"{c[0][:4]}-{c[0][4:6]}-{c[0][6:8]}T{c[1][:2]}:{c[1][2:4]}"))
                lat.append(float(c[4][:-1]) * (1 if c[4][-1] == "N" else -1))
                lon.append(float(c[5][:-1]) * (1 if c[5][-1] == "E" else -1))
                vm = float(c[6]); pm = float(c[7])
                vmax.append(np.nan if vm < 0 else vm)
                pmin.append(np.nan if pm < 0 else pm)
            storms[sid] = dict(
                name=name, time=np.array(t, dtype="datetime64[m]"),
                lat=np.array(lat), lon=np.array(lon),
                vmax=np.array(vmax), pmin=np.array(pmin))
            i += 1 + n
        else:
            i += 1
    return storms


# ======================================================================
# Geometry / interpolation / center finding
# ======================================================================
def haversine_km(lat1, lon1, lat2, lon2):
    """Great-circle distance (km). Broadcasts over array lat2/lon2."""
    p1, p2 = np.radians(lat1), np.radians(lat2)
    dp = np.radians(lat2 - lat1)
    dl = np.radians(lon2 - lon1)
    a = np.sin(dp / 2) ** 2 + np.cos(p1) * np.cos(p2) * np.sin(dl / 2) ** 2
    return 2 * EARTH_R_KM * np.arcsin(np.sqrt(np.clip(a, 0, 1)))


def to_hpa(mslp):
    """Convert MSLP to hPa if it looks like Pa (median > 2000)."""
    mslp = np.asarray(mslp, float)
    finite = mslp[np.isfinite(mslp)]
    if finite.size and np.nanmedian(finite) > 2000.0:
        return mslp / 100.0
    return mslp


def interp_to_times(src_t, src_v, dst_t):
    """Linear-in-time interpolation; NaN outside the source span / if <2 points."""
    src_v = np.asarray(src_v, float)
    m = np.isfinite(src_v)
    if m.sum() < 2:
        return np.full(len(dst_t), np.nan)
    xs = src_t[m].astype("datetime64[s]").astype("float64")
    xd = np.asarray(dst_t).astype("datetime64[s]").astype("float64")
    return np.interp(xd, xs, src_v[m], left=np.nan, right=np.nan)


def find_center(mslp2d, lat2d, lon2d, lat0, lon0, R_km=300.0, refine=False):
    """
    Locate the MSLP minimum within R_km of (lat0, lon0).
    Returns (center_lat, center_lon, min_mslp). NaN if no valid point in range.
    `refine` adds a parabolic sub-grid position estimate in index space.
    """
    if not (np.isfinite(lat0) and np.isfinite(lon0)):
        return np.nan, np.nan, np.nan
    d = haversine_km(lat0, lon0, lat2d, lon2d)
    region = (d <= R_km) & np.isfinite(mslp2d)
    if not region.any():
        return np.nan, np.nan, np.nan
    field = np.where(region, mslp2d, np.inf)
    j, i = np.unravel_index(np.argmin(field), field.shape)
    clat, clon, cmin = float(lat2d[j, i]), float(lon2d[j, i]), float(mslp2d[j, i])
    if refine and 0 < j < mslp2d.shape[0] - 1 and 0 < i < mslp2d.shape[1] - 1:
        def parab(fm, f0, fp):
            den = fm - 2 * f0 + fp
            return 0.5 * (fm - fp) / den if abs(den) > 1e-9 else 0.0
        dj = parab(mslp2d[j - 1, i], cmin, mslp2d[j + 1, i])
        di = parab(mslp2d[j, i - 1], cmin, mslp2d[j, i + 1])
        dj, di = np.clip(dj, -1, 1), np.clip(di, -1, 1)
        # bilinear position shift from neighbouring grid coordinates
        clat = clat + dj * (lat2d[min(j + 1, mslp2d.shape[0] - 1), i] - clat) * (dj > 0) \
                    + (-dj) * (lat2d[max(j - 1, 0), i] - clat) * (dj < 0)
        clon = clon + di * (lon2d[j, min(i + 1, mslp2d.shape[1] - 1)] - clon) * (di > 0) \
                    + (-di) * (lon2d[j, max(i - 1, 0)] - clon) * (di < 0)
    return clat, clon, cmin


def max_in_region(spd2d, lat2d, lon2d, lat0, lon0, R_km=300.0):
    """Max value of a field within R_km of (lat0, lon0). NaN if none valid."""
    if not (np.isfinite(lat0) and np.isfinite(lon0)) or spd2d is None:
        return np.nan
    d = haversine_km(lat0, lon0, lat2d, lon2d)
    region = (d <= R_km) & np.isfinite(spd2d)
    return float(np.max(spd2d[region])) if region.any() else np.nan


def track_series(mslp3d, spd3d, lat2d, lon2d, lat0_t, lon0_t, R_km=300.0,
                 refine=False):
    """
    Track one field through time. mslp3d:(T,Y,X); lat0_t,lon0_t:(T,) best-track.
    Returns clat, clon, cpmin, cwind  (each length T).
    """
    T = mslp3d.shape[0]
    clat = np.full(T, np.nan); clon = np.full(T, np.nan)
    cpmin = np.full(T, np.nan); cwind = np.full(T, np.nan)
    for k in range(T):
        clat[k], clon[k], cpmin[k] = find_center(
            mslp3d[k], lat2d, lon2d, lat0_t[k], lon0_t[k], R_km, refine)
        if spd3d is not None:
            cwind[k] = max_in_region(spd3d[k], lat2d, lon2d,
                                     lat0_t[k], lon0_t[k], R_km)
    return clat, clon, cpmin, cwind


def track_ensemble(mslp4d, spd4d, lat2d, lon2d, lat0_t, lon0_t, R_km=300.0,
                   refine=False):
    """Track each member. mslp4d:(M,T,Y,X). Returns (M,T) clat,clon,cpmin,cwind."""
    M, T = mslp4d.shape[:2]
    out = [np.full((M, T), np.nan) for _ in range(4)]
    for m in range(M):
        s = track_series(mslp4d[m], None if spd4d is None else spd4d[m],
                         lat2d, lon2d, lat0_t, lon0_t, R_km, refine)
        for a, v in zip(out, s):
            a[m] = v
    return out  # clat, clon, cpmin, cwind


# ======================================================================
# Per-storm tracking + error assembly
# ======================================================================
def load_window(ds, t0, t1, varnames):
    """
    Load ONLY the [t0, t1] window of `varnames` into memory, in a single read.
    The yearly array is never materialized -- only the window's time slices are
    read (integer-position selection, robust to a non-monotonic time axis).
    Returns an in-memory Dataset, or None if no timesteps fall in the window.
    """
    t = ds["time"].values
    pos = np.where((t >= t0) & (t <= t1))[0]
    if pos.size == 0:
        return None
    present = [v for v in varnames if v in ds]
    return ds[present].isel(time=pos).load()


def track_storm(name, hurdat_id, category, t0, t1, bt,
                dsT, dsU, dsE, dsG, lat2d, lon2d,
                mslp_name="WRF_MSLP", R_km=300.0, track_wind=True, refine=False):
    """
    Track one storm in all sources and assemble centers, intensities and errors.
    bt: HURDAT2 record dict for this storm. dsT/dsU/dsE/dsG: lazy per-year
    datasets (CONUS404 target / UNet / ERA5 ens / GDAS ens). Only each storm's
    [t0, t1] window is read into memory (once per source) -- never the whole
    year. Returns xr.Dataset.
    """
    t0, t1 = np.datetime64(t0), np.datetime64(t1)
    for nm, ds in (("CONUS404", dsT), ("UNet", dsU), ("ERA5", dsE), ("GDAS", dsG)):
        if mslp_name not in ds:
            raise KeyError(f"{nm} dataset has no '{mslp_name}' variable "
                           f"(set MSLP_NAME). Has: {list(ds.data_vars)}")
    needed = [mslp_name] + (["WRF_U10", "WRF_V10"] if track_wind else [])

    # ---- read ONLY this storm's window into memory, once per source ----
    wT = load_window(dsT, t0, t1, needed)
    wU = load_window(dsU, t0, t1, needed)
    wE = load_window(dsE, t0, t1, needed)
    wG = load_window(dsG, t0, t1, needed)
    if any(w is None for w in (wT, wU, wE, wG)):
        warnings.warn(f"{name}: a source has no timesteps in window {t0}..{t1}")
        return None

    # common model hours across sources (now over the small window only)
    ct = wT["time"].values
    for w in (wU, wE, wG):
        ct = np.intersect1d(ct, w["time"].values)
    if ct.size == 0:
        warnings.warn(f"{name}: sources share no common hours in {t0}..{t1}")
        return None
    wT, wU, wE, wG = (w.sel(time=ct) for w in (wT, wU, wE, wG))   # in-memory align

    # best-track interpolated to model hours
    bt_lat = interp_to_times(bt["time"], bt["lat"], ct)
    bt_lon = interp_to_times(bt["time"], bt["lon"], ct)
    bt_vmax = interp_to_times(bt["time"], bt["vmax"], ct)
    bt_pmin = interp_to_times(bt["time"], bt["pmin"], ct)

    def mslp_of(w, ens=False):
        da = w[mslp_name].transpose("member", "time", ...) if ens \
            else w[mslp_name].transpose("time", ...)
        return to_hpa(da.values)

    def wind_of(w, ens=False):
        if not track_wind:
            return None
        order = ("member", "time", ...) if ens else ("time", ...)
        return np.sqrt(w["WRF_U10"].transpose(*order).values ** 2
                       + w["WRF_V10"].transpose(*order).values ** 2)

    # CONUS404 + UNet (deterministic)
    c4 = track_series(mslp_of(wT), wind_of(wT), lat2d, lon2d,
                      bt_lat, bt_lon, R_km, refine)
    un = track_series(mslp_of(wU), wind_of(wU), lat2d, lon2d,
                      bt_lat, bt_lon, R_km, refine)
    # ensembles
    e5 = track_ensemble(mslp_of(wE, True), wind_of(wE, True), lat2d, lon2d,
                        bt_lat, bt_lon, R_km, refine)
    gd = track_ensemble(mslp_of(wG, True), wind_of(wG, True), lat2d, lon2d,
                        bt_lat, bt_lon, R_km, refine)
    M = e5[0].shape[0]

    ds = xr.Dataset(
        coords=dict(time=ct, member=np.arange(M)),
        data_vars=dict(
            bt_lat=("time", bt_lat), bt_lon=("time", bt_lon),
            bt_vmax=("time", bt_vmax), bt_pmin=("time", bt_pmin),
            c404_lat=("time", c4[0]), c404_lon=("time", c4[1]),
            c404_pmin=("time", c4[2]), c404_wind=("time", c4[3]),
            unet_lat=("time", un[0]), unet_lon=("time", un[1]),
            unet_pmin=("time", un[2]), unet_wind=("time", un[3]),
            era5_lat=(("member", "time"), e5[0]), era5_lon=(("member", "time"), e5[1]),
            era5_pmin=(("member", "time"), e5[2]), era5_wind=(("member", "time"), e5[3]),
            gdas_lat=(("member", "time"), gd[0]), gdas_lon=(("member", "time"), gd[1]),
            gdas_pmin=(("member", "time"), gd[2]), gdas_wind=(("member", "time"), gd[3]),
        ),
        attrs=dict(storm=name, hurdat_id=hurdat_id, category=int(category),
                   R_km=R_km, mslp_units="hPa"),
    )

    # --- track error (great-circle) vs HURDAT2 and vs CONUS404 ---
    def trkerr(la, lo, rla, rlo):
        return haversine_km(rla, rlo, la, lo)
    ds["c404_trkerr_hd"] = ("time", trkerr(c4[0], c4[1], bt_lat, bt_lon))
    ds["unet_trkerr_hd"] = ("time", trkerr(un[0], un[1], bt_lat, bt_lon))
    ds["era5_trkerr_hd"] = (("member", "time"),
                            trkerr(e5[0], e5[1], bt_lat[None, :], bt_lon[None, :]))
    ds["gdas_trkerr_hd"] = (("member", "time"),
                            trkerr(gd[0], gd[1], bt_lat[None, :], bt_lon[None, :]))
    ds["unet_trkerr_c4"] = ("time", trkerr(un[0], un[1], c4[0], c4[1]))
    ds["era5_trkerr_c4"] = (("member", "time"),
                            trkerr(e5[0], e5[1], c4[0][None, :], c4[1][None, :]))
    ds["gdas_trkerr_c4"] = (("member", "time"),
                            trkerr(gd[0], gd[1], c4[0][None, :], c4[1][None, :]))

    # --- ensemble-MEAN position error (distinct from mean of member errors) ---
    for ens, arr in (("era5", e5), ("gdas", gd)):
        mlat, mlon = np.nanmean(arr[0], 0), np.nanmean(arr[1], 0)
        ds[f"{ens}_ensmean_lat"] = ("time", mlat)
        ds[f"{ens}_ensmean_lon"] = ("time", mlon)
        ds[f"{ens}_ensmean_trkerr_hd"] = ("time", trkerr(mlat, mlon, bt_lat, bt_lon))

    # --- central-pressure error (source - reference) ---
    ds["c404_pminerr_hd"] = ("time", c4[2] - bt_pmin)
    ds["unet_pminerr_hd"] = ("time", un[2] - bt_pmin)
    ds["era5_pminerr_hd"] = (("member", "time"), e5[2] - bt_pmin[None, :])
    ds["gdas_pminerr_hd"] = (("member", "time"), gd[2] - bt_pmin[None, :])
    ds["unet_pminerr_c4"] = ("time", un[2] - c4[2])
    ds["era5_pminerr_c4"] = (("member", "time"), e5[2] - c4[2][None, :])
    ds["gdas_pminerr_c4"] = (("member", "time"), gd[2] - c4[2][None, :])

    # --- max-wind error vs HURDAT2 Vmax (secondary; mean wind vs 1-min sustained) ---
    if track_wind:
        ds["c404_winderr_hd"] = ("time", c4[3] - bt_vmax)
        ds["unet_winderr_hd"] = ("time", un[3] - bt_vmax)
        ds["era5_winderr_hd"] = (("member", "time"), e5[3] - bt_vmax[None, :])
        ds["gdas_winderr_hd"] = (("member", "time"), gd[3] - bt_vmax[None, :])
    return ds


# ======================================================================
# Aggregation across storms (combine all cases)
# ======================================================================
def summarize(storm_dsets, categories=None):
    """
    Pool track / central-pressure errors across all storms (and members) and
    print a summary per source. categories: optional iterable to filter storms.
    Returns a dict of pooled arrays.
    """
    dsets = [d for d in storm_dsets if d is not None]
    if categories is not None:
        dsets = [d for d in dsets if d.attrs["category"] in set(categories)]
    pool = {}

    def grab(key):
        vals = []
        for d in dsets:
            if key in d:
                vals.append(np.asarray(d[key].values).ravel())
        return np.concatenate(vals) if vals else np.array([])

    print("\n" + "#" * 72)
    tag = f"cat {sorted(categories)}" if categories else "all cases"
    print(f" TRACK & CENTRAL-PRESSURE ERRORS  ({len(dsets)} storms, {tag})")
    print("#" * 72)
    print(f" {'source':<22s}{'N':>7s}{'trk med':>9s}{'trk mean':>9s}"
          f"{'trk p90':>9s}{'dPmin mean':>11s}{'dPmin sd':>9s}")

    def line(label, trk_key, pmin_key):
        trk = grab(trk_key); trk = trk[np.isfinite(trk)]
        pm = grab(pmin_key); pm = pm[np.isfinite(pm)]
        pool[label] = dict(trk=trk, dpmin=pm)
        if trk.size == 0:
            print(f" {label:<22s}{'-':>7s}"); return
        print(f" {label:<22s}{trk.size:>7d}{np.median(trk):>9.1f}"
              f"{np.mean(trk):>9.1f}{np.percentile(trk, 90):>9.1f}"
              f"{np.mean(pm):>11.1f}{np.std(pm):>9.1f}")

    # vs HURDAT2
    print(" -- vs HURDAT2 (absolute) --")
    line("CONUS404", "c404_trkerr_hd", "c404_pminerr_hd")
    line("UNet", "unet_trkerr_hd", "unet_pminerr_hd")
    line("ERA5 members", "era5_trkerr_hd", "era5_pminerr_hd")
    line("GDAS members", "gdas_trkerr_hd", "gdas_pminerr_hd")
    line("ERA5 ens-mean pos", "era5_ensmean_trkerr_hd", "era5_pminerr_hd")
    line("GDAS ens-mean pos", "gdas_ensmean_trkerr_hd", "gdas_pminerr_hd")
    # vs CONUS404 (downscaling fidelity)
    print(" -- vs CONUS404 (downscaling) --")
    line("UNet", "unet_trkerr_c4", "unet_pminerr_c4")
    line("ERA5 members", "era5_trkerr_c4", "era5_pminerr_c4")
    line("GDAS members", "gdas_trkerr_c4", "gdas_pminerr_c4")
    print("\n (trk = track error km;  dPmin = central-pressure bias hPa, "
          "source - reference)")
    return pool


# ======================================================================
# MAIN
# ======================================================================
if __name__ == "__main__":
    from glob import glob

    # -------------------- config --------------------
    base = "/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_TC"
    OUTDIR = f"{base}/tc_track_intensity"
    HURDAT_CACHE = f"{OUTDIR}/hurdat2_atlantic.txt"
    MSLP_NAME = "WRF_MSLP"
    R_KM = 300.0              # center search radius around best-track
    TRACK_WIND = True         # also track max 10 m wind (secondary, caveated)
    YEARS = [2020, 2021, 2022, 2023, 2024]
    os.makedirs(OUTDIR, exist_ok=True)

    raw_cases = [
        # season, hurdat_id, storm, category, track_start, track_end
        (2020, "AL082020", "Hanna",    1, "2020-07-23 00:00", "2020-07-26 00:00"),
        (2020, "AL092020", "Isaias",   1, "2020-07-31 06:00", "2020-08-04 18:00"),
        (2020, "AL142020", "Marco",    1, "2020-08-22 12:00", "2020-08-25 00:00"),
        (2020, "AL132020", "Laura",    4, "2020-08-24 02:00", "2020-08-29 00:00"),
        (2020, "AL192020", "Sally",    2, "2020-09-11 18:00", "2020-09-17 06:00"),
        (2020, "AL252020", "Gamma",    1, "2020-10-03 16:45", "2020-10-06 12:00"),
        (2020, "AL262020", "Delta",    3, "2020-10-07 06:00", "2020-10-10 12:00"),
        (2020, "AL282020", "Zeta",     3, "2020-10-27 03:55", "2020-10-29 12:00"),
        (2020, "AL292020", "Eta",      1, "2020-11-08 00:00", "2020-11-13 06:00"),
        (2021, "AL052021", "Elsa",     1, "2021-07-05 00:00", "2021-07-09 16:30"),
        (2021, "AL082021", "Henri",    1, "2021-08-18 18:00", "2021-08-23 12:00"),
        (2021, "AL072021", "Grace",    3, "2021-08-19 09:45", "2021-08-21 12:00"),
        (2021, "AL092021", "Ida",      4, "2021-08-27 12:00", "2021-09-01 06:00"),
        (2021, "AL142021", "Nicholas", 1, "2021-09-12 12:00", "2021-09-15 12:00"),
        (2022, "AL072022", "Fiona",    4, "2022-09-20 00:00", "2022-09-23 06:00"),
        (2022, "AL092022", "Ian",      5, "2022-09-27 00:00", "2022-09-30 18:05"),
        (2022, "AL172022", "Nicole",   1, "2022-11-07 06:00", "2022-11-11 12:00"),
        (2023, "AL082023", "Franklin", 4, "2023-08-24 00:00", "2023-08-30 12:00"),
        (2023, "AL102023", "Idalia",   4, "2023-08-26 12:00", "2023-08-31 06:00"),
        (2023, "AL132023", "Lee",      2, "2023-09-13 12:00", "2023-09-15 12:00"),
        (2024, "AL022024", "Beryl",    1, "2024-07-05 11:00", "2024-07-09 06:00"),
        (2024, "AL042024", "Debby",    1, "2024-08-03 00:00", "2024-08-08 18:00"),
        (2024, "AL052024", "Ernesto",  2, "2024-08-14 18:00", "2024-08-16 06:00"),
        (2024, "AL062024", "Francine",  2, "2024-09-09 12:00", "2024-09-12 12:00"),
        (2024, "AL092024", "Helene",   4, "2024-09-25 06:00", "2024-09-27 12:00"),
    ]

    # -------------------- HURDAT2 --------------------
    storms_bt = parse_hurdat2(download_hurdat2(cache_path=HURDAT_CACHE))

    # -------------------- static grid --------------------
    ds_static = xr.open_zarr(f"{base}/static/C404_TC_static_8km.zarr")
    lat2d = ds_static["XLAT"].values
    lon2d = ds_static["XLONG"].values

    def natkey(s):
        return [int(t) if t.isdigit() else t for t in re.split(r"(\d+)", s)]

    def uv_keep(ds):                       # keep MSLP + 10 m wind components
        keep = [v for v in (MSLP_NAME, "WRF_U10", "WRF_V10") if v in ds]
        return ds[keep]

    def load_ensemble(pattern):
        files = sorted(glob(pattern), key=natkey)
        members = [uv_keep(xr.open_zarr(f)) for f in files]
        ds = xr.concat(members, dim="member")
        return ds.assign_coords(member=np.arange(ds.sizes["member"]))

    cases_by_year = {}
    for season, hid, name, cat, ta, tb in raw_cases:
        cases_by_year.setdefault(season, []).append((hid, name, cat, ta, tb))

    all_storm_ds = []
    for year in YEARS:
        if year not in cases_by_year:
            continue
        print(f"\n===== {year} =====")
        # open the year datasets LAZILY (metadata only); track_storm reads just
        # each storm's [track_start, track_end] window into memory.
        dsT = uv_keep(xr.open_zarr(f"{base}/C404_CorrDiff/TC_target_{year}.zarr"))
        dsU = uv_keep(xr.open_zarr(f"{base}/TC_UNET/TC_UNET_pred_{year}_MSLP.zarr"))
        dsE = load_ensemble(
            f"{base}/TC_pred_corrdiff_final/TC_ERA5_corrdiff_pred_{year}_mem*.zarr")
        dsG = load_ensemble(
            f"{base}/TC_pred_corrdiff_final/TC_GDAS_corrdiff_pred_{year}_mem*.zarr")
        for hid, name, cat, ta, tb in cases_by_year[year]:
            if hid not in storms_bt:
                warnings.warn(f"{name} ({hid}) not found in HURDAT2 — skipping")
                continue
            sd = track_storm(name, hid, cat, ta, tb, storms_bt[hid],
                             dsT, dsU, dsE, dsG, lat2d, lon2d,
                             mslp_name=MSLP_NAME, R_km=R_KM, track_wind=TRACK_WIND)
            if sd is None:
                continue
            nt = sd.sizes["time"]
            tehd = float(np.nanmedian(sd["era5_trkerr_hd"].values))
            print(f"  {name:<10s} cat{cat}  {nt:3d} hrs tracked  "
                  f"ERA5 median trk err vs HURDAT2 = {tehd:.0f} km")
            sd.to_netcdf(f"{OUTDIR}/{year}_{name}.nc")
            all_storm_ds.append(sd)

    # -------------------- combine all cases --------------------
    summarize(all_storm_ds)
    summarize(all_storm_ds, categories={3, 4, 5})   # majors only
    print(f"\nPer-storm tracks written to {OUTDIR}/<year>_<storm>.nc")

downloading HURDAT2 -> /glade/derecho/scratch/ksha/DWC_data/CONUS_domain_TC/tc_track_intensity/hurdat2_atlantic.txt

===== 2020 =====
  Hanna      cat1   73 hrs tracked  ERA5 median trk err vs HURDAT2 = 59 km
  Isaias     cat1  109 hrs tracked  ERA5 median trk err vs HURDAT2 = 39 km
  Marco      cat1   61 hrs tracked  ERA5 median trk err vs HURDAT2 = 58 km
  Laura      cat4  119 hrs tracked  ERA5 median trk err vs HURDAT2 = 37 km
  Sally      cat2  133 hrs tracked  ERA5 median trk err vs HURDAT2 = 51 km
  Gamma      cat1   68 hrs tracked  ERA5 median trk err vs HURDAT2 = 47 km
  Delta      cat3   79 hrs tracked  ERA5 median trk err vs HURDAT2 = 24 km
  Zeta       cat3   57 hrs tracked  ERA5 median trk err vs HURDAT2 = 32 km
  Eta        cat1  127 hrs tracked  ERA5 median trk err vs HURDAT2 = 75 km

===== 2021 =====
  Elsa       cat1  113 hrs tracked  ERA5 median trk err vs HURDAT2 = 98 km


/glade/derecho/scratch/ksha/tmp/ipykernel_71757/722994539.py:325: RuntimeWarning: Mean of empty slice
  mlat, mlon = np.nanmean(arr[0], 0), np.nanmean(arr[1], 0)


  Henri      cat1  115 hrs tracked  ERA5 median trk err vs HURDAT2 = 48 km
  Grace      cat3   51 hrs tracked  ERA5 median trk err vs HURDAT2 = 32 km
  Ida        cat4  115 hrs tracked  ERA5 median trk err vs HURDAT2 = 65 km
  Nicholas   cat1   73 hrs tracked  ERA5 median trk err vs HURDAT2 = 87 km

===== 2022 =====


/glade/derecho/scratch/ksha/tmp/ipykernel_71757/722994539.py:325: RuntimeWarning: Mean of empty slice
  mlat, mlon = np.nanmean(arr[0], 0), np.nanmean(arr[1], 0)


  Fiona      cat4   79 hrs tracked  ERA5 median trk err vs HURDAT2 = 93 km
  Ian        cat5   91 hrs tracked  ERA5 median trk err vs HURDAT2 = 25 km


...

  Franklin   cat4  157 hrs tracked  ERA5 median trk err vs HURDAT2 = 57 km
  Idalia     cat4  115 hrs tracked  ERA5 median trk err vs HURDAT2 = 34 km


/glade/derecho/scratch/ksha/tmp/ipykernel_71757/722994539.py:325: RuntimeWarning: Mean of empty slice
  mlat, mlon = np.nanmean(arr[0], 0), np.nanmean(arr[1], 0)


  Lee        cat2   49 hrs tracked  ERA5 median trk err vs HURDAT2 = 186 km

===== 2024 =====
  Beryl      cat1   92 hrs tracked  ERA5 median trk err vs HURDAT2 = 24 km
  Debby      cat1  139 hrs tracked  ERA5 median trk err vs HURDAT2 = 32 km


/glade/derecho/scratch/ksha/tmp/ipykernel_71757/722994539.py:325: RuntimeWarning: Mean of empty slice
  mlat, mlon = np.nanmean(arr[0], 0), np.nanmean(arr[1], 0)


  Ernesto    cat2   37 hrs tracked  ERA5 median trk err vs HURDAT2 = 299 km
  Francine   cat2   73 hrs tracked  ERA5 median trk err vs HURDAT2 = 45 km
  Helene     cat4   55 hrs tracked  ERA5 median trk err vs HURDAT2 = 27 km

########################################################################
 TRACK & CENTRAL-PRESSURE ERRORS  (25 storms, all cases)
########################################################################
 source                      N  trk med trk mean  trk p90 dPmin mean dPmin sd
 -- vs HURDAT2 (absolute) --
 CONUS404                 2139     60.0     83.0    189.5        8.6     12.8
 UNet                     2139     36.8     55.2    123.1       12.3     13.3
 ERA5 members            42780     45.5     63.4    135.3       12.9     13.3
 GDAS members            42780     52.3     71.3    154.3       14.1     13.6
 ERA5 ens-mean pos        2139     37.1     55.0    117.2       12.9     13.3
 GDAS ens-mean pos        2139     46.3     63.7    138.4       14.1     